In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
data = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
data_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
data_id.columns = data_id.columns.str.replace("-", "_")
data = data.merge(data_id, how='left', on='TransactionID')

In [4]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score, f1_score, classification_report
import category_encoders as ce
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
df = train.merge(identity, how='left', on='TransactionID')
from sklearn.model_selection import train_test_split

X = df.drop(columns=["isFraud"])
y = df["isFraud"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
class InfCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        return X

class DropHighNaN(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class DropIDColumns(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.cols_to_drop_ = [c for c in X.columns if "id" in c.lower()]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class DropNearConstant(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.99):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = [
            col for col in X.columns
            if X[col].value_counts(normalize=True, dropna=False).iloc[0] > self.threshold
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class DropCorrelated(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        num_X = X.select_dtypes(include=np.number)
        corr = num_X.corr().abs()

        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.cols_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class LogSkewTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        num = X.select_dtypes(include=np.number)
        self.skew_cols_ = num.columns[num.skew().abs() > self.threshold]
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.skew_cols_:
            # replace inf just in case
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)

            # fill NaN with 0 (safe baseline)
            X[col] = X[col].fillna(0)

            # handle negative values safely
            min_val = X[col].min()
            if min_val < 0:
                X[col] = X[col] - min_val  # shift to make all >= 0

            # now safe log transform
            X[col] = np.log1p(X[col])

        return X

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.rare_maps_ = {}

        cat_cols = X.select_dtypes(include="object").columns

        for col in cat_cols:
            freq = X[col].value_counts(normalize=True)
            self.rare_maps_[col] = freq[freq < self.threshold].index

        return self

    def transform(self, X):
        X = X.copy()

        for col, rare_vals in self.rare_maps_.items():
            X[col] = X[col].replace(rare_vals, "Other")

        return X


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.keep_columns_ = None

    def fit(self, X, y=None):
        # Convert to DataFrame if necessary
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()

        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        to_drop = [
            column for column in upper.columns
            if any(upper[column] > self.threshold)
        ]

        self.keep_columns_ = [
            col for col in X.columns
            if col not in to_drop
        ]

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        return X[self.keep_columns_]

class RFESelector(BaseEstimator, TransformerMixin):
    def __init__(self, estimator, n_features_to_select=30):
        self.estimator = estimator
        self.n_features_to_select = n_features_to_select

    def fit(self, X, y):
        self.rfe_ = RFE(
            estimator=self.estimator,
            n_features_to_select=self.n_features_to_select
        )
        self.rfe_.fit(X, y)
        return self

    def transform(self, X):
        return self.rfe_.transform(X)
        

class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, model, threshold=0.5):
        self.model = model
        self.threshold = threshold

    def fit(self, X, y):
        self.model.fit(X, y)
        self.classes_ = self.model.classes_
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= self.threshold).astype(int)

import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
# ---------------- PRE-CLEANING ----------------
pre_cleaning = Pipeline(steps=[
    ("inf_clean", InfCleaner()),
    ("drop_nan", DropHighNaN()),
    ("drop_id", DropIDColumns()),
    ("drop_const", DropNearConstant()),
    #("log_skew", LogSkewTransformer()),
    ("rare_groups", RareCategoryGrouper()),
])


# Fit only on training data to discover final columns
X_clean = pre_cleaning.fit_transform(X_train)


# Detect feature types AFTER cleaning
num_cols = X_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_clean.select_dtypes(include=["object", "category"]).columns.tolist()


# ---------------- NUMERIC PIPELINE ----------------
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
    #("scaler", StandardScaler())
])


# ---------------- CATEGORICAL PIPELINE ----------------
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", ce.TargetEncoder())
])


# ---------------- FEATURE PIPELINE ----------------
feature_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)
xgb =XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight= 27.57,  # IMPORTANT for imbalance
    random_state=42,
    eval_metric="logloss"
)

# ---------------- FINAL MODEL PIPELINE ----------------
model = Pipeline(steps=[
    ("pre_clean", pre_cleaning),
    ("features", feature_pipeline),
    #("corr_filter", CorrelationFilter(threshold=0.85)),
    #("feature_selection", SelectKBest(score_func=f_classif, k=50)),
    ("clf", ThresholdClassifier(
        model=xgb,
        threshold=0.83
    ))])

model.fit(X_train, y_train)

Pipeline(steps=[('pre_clean',
                 Pipeline(steps=[('inf_clean', InfCleaner()),
                                 ('drop_nan', DropHighNaN()),
                                 ('drop_id', DropIDColumns()),
                                 ('drop_const', DropNearConstant()),
                                 ('rare_groups', RareCategoryGrouper())])),
                ('features',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['TransactionDT',
                                                   'TransactionAmt',...
                                                         gamma=None,
                                                         grow_policy=None,
                                                         importance_type=None,
                                                         interaction_constraints=None,
                                                         learning_rate=0.05,
                                                         max_bin=None,
                                                         max_cat_threshold=None,
                                                         max_cat_to_onehot=None,
                                                         max_delta_step=None,
                                                         max_depth=6,
                                                         max_leaves=None,
                                                         min_child_weight=None,
                                                         missing=nan,
                                                         monotone_constraints=None,
                                                         multi_strategy=None,
                                                         n_estimators=300,
                                                         n_jobs=None,
                                                         num_parallel_tree=None, ...),
                                     threshold=0.83))])

In [5]:
predictions = model.predict(data)

print(predictions)

[0 0 0 ... 0 0 0]


In [6]:
submission_df = pd.DataFrame({
    'TransactionID': data['TransactionID'], 
    'isFraud': predictions  
})
submission_df.to_csv('submission.csv', index=False)

print("Submission file generated!")

Submission file generated!
